In [42]:
import os
import pandas as pd
import numpy as np

PATH_TO_DATA       = os.path.join(".", "ucr_datasets")
PATH_TO_METADATA   = os.path.join(".", "benchmarks", 'metadata.pkl')
PATH_TO_BENCHMARKS = os.path.join(".", "benchmarks")

# number time series per benchmark
NB_SERIES  = 250
# Percentage train
TRAIN_SIZE = 0.20

In [43]:
def get_kappa_max(nclasses):
    return int(np.floor((nclasses+1)/3))

def distance_matrix(time_series, distance_fun):
    n = len(time_series)
    D = np.full((n, n), np.inf)
    for i in range(n):
        for j in range(i, n):
            d = distance_fun(time_series[i], time_series[j])
            D[i, j] = D[j, i] = d
    return D
    
def znormalized_squared_ed(series1, series2):
    series1 = (series1 - np.mean(series1, axis=None)) / np.std(series1, axis=None)
    series2 = (series2 - np.mean(series2, axis=None)) / np.std(series2, axis=None)
    d = np.sum(np.power(series1 - series2, 2))
    return d

def calculate_radius(instances):
    D = distance_matrix(instances, znormalized_squared_ed)
    return np.ceil(np.amin(np.amax(D, axis=0)))

In [44]:
VAR_LENGTH_DATASETS = ["CharacterTrajectories", "SpokenArabicDigits", "JapaneseVowels"]
FIX_LENGTH_DATASETS = ["ArticularyWordRecognition", "ERing", "Plane", "Cricket", "Mallat", "UWaveGestureLibrary", "Symbols", "PenDigits", "Fungi", "NATOPS", "ECG5000"]

In [55]:
import generation

columns = {'ds_name': str, 'nclasses': int, 'multivariate': bool, 'variable_length': bool, 'l_min': int, 'l_max': int, 'radius': int}
metadata = pd.DataFrame(columns, index=[])
metadata['multivariate']    = metadata['multivariate'].astype(bool)
metadata['variable_length'] = metadata['variable_length'].astype(bool)

for ds_name in FIX_LENGTH_DATASETS + VAR_LENGTH_DATASETS:
    print(ds_name)
    path_to_dataset = os.path.join(PATH_TO_DATA, ds_name.lower())
    df = pd.read_pickle(os.path.join(path_to_dataset, "znormalized.pkl"))
    
    np.random.seed(0)

    # Resplit
    df_train = df.groupby('label', group_keys=False).apply(lambda x: x.sample(frac=TRAIN_SIZE)).sample(frac=1.0).reset_index(drop=True)
    df_test  = df.drop(df_train.index).sample(frac=1.0).reset_index(drop=True)
         
    # Store the train and test set of instances
    df_train.to_pickle(os.path.join(path_to_dataset, 'train.pkl'))
    df_test.to_pickle(os.path.join(path_to_dataset, 'test.pkl'))

    # Generate tsmd benchmark
    classes = df['label'].unique()
    nclasses  = len(classes)
    kappa_max = get_kappa_max(nclasses)
    
    nb_train = int(TRAIN_SIZE * NB_SERIES) 
    nb_test  = NB_SERIES - nb_train
    
    benchmark_train = generation.generate_tsmd_benchmark(df_train, nb_train, kappa_max)
    benchmark_test  = generation.generate_tsmd_benchmark(df_test,  nb_test,  kappa_max)
    
    # Store the benchmark
    path_to_benchmark = os.path.join(PATH_TO_BENCHMARKS, ds_name.lower())
    
    if not os.path.exists(os.path.join(path_to_benchmark, 'train.pkl')):
        benchmark_train.to_pickle(os.path.join(path_to_benchmark, 'train.pkl'))
        benchmark_test.to_pickle(os.path.join(path_to_benchmark, 'test.pkl'))
        
    # Store metadata about the instances in the train set
    univariate = df_train['series'].iloc[0].shape[1] == 1
    lengths = df_train['length'].to_numpy()
    l_min, l_max = np.min(lengths), np.max(lengths)
    fixed_length = l_min == l_max
    
    radius = np.nan
    if fixed_length and univariate:
        df_radii = df_train.groupby('label')['series'].apply(list).apply(calculate_radius)
        radius = df_radii.max()
    
    new_row = {'ds_name': ds_name.lower(), 'nclasses': nclasses, 'multivariate': not univariate, 'variable_length': not fixed_length, 'l_min': l_min, 'l_max': l_max, 'radius': radius}
    metadata = pd.concat((metadata, pd.DataFrame([new_row])))

ArticularyWordRecognition
ERing
Plane
Cricket
Mallat
UWaveGestureLibrary
Symbols
PenDigits
Fungi
NATOPS
ECG5000
CharacterTrajectories
SpokenArabicDigits
JapaneseVowels


In [56]:
metadata = metadata.reset_index(drop=True)
metadata

,ds_name,nclasses,multivariate,variable_length,l_min,l_max,radius
0,articularywordrecognition,25,True,False,144,144,NaN
1,ering,6,True,False,65,65,NaN
2,plane,7,False,False,144,144,17.0
3,cricket,12,True,False,1197,1197,NaN
4,mallat,8,False,False,1024,1024,118.0
5,uwavegesturelibrary,8,True,False,315,315,NaN
6,symbols,6,False,False,398,398,973.0
7,pendigits,10,True,False,8,8,NaN
8,fungi,18,False,False,201,201,43.0
9,natops,6,True,False,51,51,NaN


In [58]:
metadata.to_pickle(os.path.join(PATH_TO_BENCHMARKS, 'metadata.pkl'))